In [218]:
import pypsa
import pandas as pd
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
import seaborn as sns
import re

In [219]:
from _helpers_notebooks import mock_snakemake
snakemake = mock_snakemake(
            "collect_figures",
            scenario="penalty-oc", # penalty-sa, penalty-ea, penalty-nwa, penalty-oc
            sort="test", # TODO update
            demand="False",
    )

In [220]:
scenario = snakemake.wildcards.scenario
mga_scenario = "mga-eur-nwa-2"

### Paths

In [ ]:
trade_model = f"../../results/cost_year~2050/interone~hbi/intertwo~eaf-grid/final~steel/scenario~{scenario}/network.nc"
trade_model_mga = f"../../results/cost_year~2050/interone~hbi/intertwo~eaf-grid/final~steel/scenario~{mga_scenario}/network.nc"

### Read trade model

In [222]:
n = pypsa.Network(trade_model)
n_mga = pypsa.Network(trade_model_mga)

INFO:pypsa.network.io:New version 1.1.2 available! (Current: 0.35.0)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 0.35.0)
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, links, loads


### Get prices

In [223]:
# Steel price analysis
steel_price = n.buses_t.marginal_price.loc[:, n.buses[n.buses.carrier == "steel"].index]
steel_price

Bus,Europe_steel,Far_West_Europe_steel,Middle_East_steel,North_West_Africa_steel,Subsaharan_Africa_steel,North_America_steel,Eurasia_steel,South_America_steel,South_South_America_steel,Central_America_steel,West_Asia_steel,East_Asia_steel,Pacific_Asia_steel,East_East_Asia_steel,Oceania_steel
snapshot,,,,,,,,,,,,,,,
now,681.470691,698.448266,684.455585,670.140798,660.34864,693.413217,699.912531,670.78762,665.533455,714.621799,702.598727,735.03076,699.663304,710.823705,636.035841


In [224]:
# Steel price analysis
steel_price_mga_presolve = n_mga.buses_t.marginal_price.loc[:, n_mga.buses[n_mga.buses.carrier == "steel"].index]
steel_price_mga_presolve

Bus,Europe_steel,Far_West_Europe_steel,Middle_East_steel,North_West_Africa_steel,Subsaharan_Africa_steel,North_America_steel,Eurasia_steel,South_America_steel,South_South_America_steel,Central_America_steel,West_Asia_steel,East_Asia_steel,Pacific_Asia_steel,East_East_Asia_steel,Oceania_steel
snapshot,,,,,,,,,,,,,,,
now,19.586335,19.046576,18.664998,18.274636,18.007605,18.909271,19.086506,18.292274,18.148994,19.487626,19.159758,20.044175,19.98307,19.384052,18.948917


### Resolve MGA model with fixed capacties

In [225]:
# After MGA run, store optimal capacities
mga_capacities = {c: n_mga.df(c)[["p_nom_opt"]].copy() 
                  for c in n_mga.branch_components | n_mga.one_port_components
                  if "p_nom_opt" in n_mga.df(c).columns}

# Fix p_nom to p_nom_opt
for c, df in mga_capacities.items():
    n_mga.df(c)["p_nom"] = df["p_nom_opt"]*1.000000001
    n_mga.df(c)["p_nom_extendable"] = False

In [226]:
solver_name = snakemake.config["solver"]["name"]
options = snakemake.config["solver_options"][snakemake.config["solver"]["options"]]
# Re-solve dispatch only (no investment)
n_mga.optimize(n_mga.snapshots, solver_name=solver_name, solver_options=options)

INFO:linopy.model: Solve problem using Gurobi solver
INFO:linopy.model:Solver options:
 - threads: 4
 - method: 2
 - crossover: 0
 - BarConvTol: 1e-06
 - Seed: 123
 - AggFill: 0
 - PreDual: 0
 - GURO_PAR_BARDENSETHRESH: 200
INFO:linopy.io: Writing time: 0.03s


Set parameter Username


INFO:gurobipy:Set parameter Username


Set parameter LicenseID to value 2797620


INFO:gurobipy:Set parameter LicenseID to value 2797620


...

INFO:gurobipy:Barrier solved model in 15 iterations and 0.07 seconds (0.00 work units)


Optimal objective 1.19411297e+12


INFO:gurobipy:Optimal objective 1.19411297e+12


INFO:gurobipy:
INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 1065 primals, 2175 duals
Objective: 1.19e+12
Solver model: available
Solver message: 2

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Link-fix-p-lower, Link-fix-p-upper were not assigned to the network.


('ok', 'optimal')

In [227]:

# Now marginal prices are meaningful again
steel_price_mga = n_mga.buses_t.marginal_price.loc[
    :, n_mga.buses[n.buses.carrier == "steel"].index
]

steel_price_mga

Bus,Europe_steel,Far_West_Europe_steel,Middle_East_steel,North_West_Africa_steel,Subsaharan_Africa_steel,North_America_steel,Eurasia_steel,South_America_steel,South_South_America_steel,Central_America_steel,West_Asia_steel,East_Asia_steel,Pacific_Asia_steel,East_East_Asia_steel,Oceania_steel
snapshot,,,,,,,,,,,,,,,
now,718.233706,698.441459,684.448799,670.13503,651.7044,693.407454,699.905722,670.78344,665.529275,714.615007,702.590919,735.022894,732.78216,710.81588,690.367153


### Price difference

In [228]:
(steel_price_mga - steel_price).round(2)

Bus,Europe_steel,Far_West_Europe_steel,Middle_East_steel,North_West_Africa_steel,Subsaharan_Africa_steel,North_America_steel,Eurasia_steel,South_America_steel,South_South_America_steel,Central_America_steel,West_Asia_steel,East_Asia_steel,Pacific_Asia_steel,East_East_Asia_steel,Oceania_steel
snapshot,,,,,,,,,,,,,,,
now,36.76,-0.01,-0.01,-0.01,-8.64,-0.01,-0.01,-0.0,-0.0,-0.01,-0.01,-0.01,33.12,-0.01,54.33
